# Optimiser en Julia avec JuMP
**Journée Julia pour les statistiques et science des données — Groupe Calcul CNRS**  
Xavier Gandibleux — Nantes Université — 15 Juin 2026

---
## Installation des packages

À faire **une seule fois** pour ajouter les packages à la distribution Julia.

In [ ]:
using Pkg

Pkg.add("JuMP")
Pkg.add("GLPK")
Pkg.add("MultiObjectiveAlgorithms")

Pkg.add("Plots")
Pkg.add("LaTeXStrings")


---
## Partie 1 — Modélisation explicite avec JuMP

### Le problème d'optimisation

$$
\begin{array}{llrcrrl}
\max        & z(x) = & x_1 & +     & 3x_2 &        &      \\
\text{s.t.} &        & x_1 & +     &  x_2 & \leq 14 & \qquad (1) \\
            &        &-2x_1 & +    & 3x_2 & \leq 12 & \qquad (2) \\
            &        & 2x_1 & -    &  x_2 & \leq 12 & \qquad (3) \\
            &        & x_1  & ,\, & x_2  & \geq 0  & \qquad (4)
\end{array}
$$

### Créer un modèle

Syntaxe : `model = Model(solver)`

### Définir les variables

Syntaxe : `@variable(model, variable)`  
Par défaut, les variables sont **continues** et **non bornées**.

### Définir la fonction objectif

Syntaxe : `@objective(model, sense, expression)`

### Définir les contraintes

Syntaxe : `@constraint(model, id, constraint)`

### Résoudre le problème

Syntaxe : `optimize!(model)`

### Extraire les résultats

### Complément : types de variables disponibles

```julia
@variable(model, x)              # x libre (non borné)
@variable(model, x >= lb)        # x borné inférieurement
@variable(model, x <= ub)        # x borné supérieurement
@variable(model, lb <= x <= ub)  # x doublement borné
@variable(model, x == 2)         # x fixé
@variable(model, x >= 0, Int)    # x entier
@variable(model, x, Bin)         # x binaire {0,1}
```

---
## Partie 2 — Modélisation implicite avec JuMP

La modélisation implicite permet de décrire un modèle à partir de **données** (vecteurs, matrices) plutôt que d'écrire chaque contrainte explicitement.

### Modélisation implicite — variables et objectif

Syntaxe pour un vecteur de variables : `@variable(model, x[1:n] >= 0)`  

### Données du problème

### Modèle JuMP et résolution

### Extraire les résultats

---
## Partie 3 — Optimisation multi-objectif avec MultiObjectiveAlgorithms

### Le problème bi-objectif

$$
\begin{array}{llrcrrl}
\max & z_1 = & x_1  & +  & x_2  &           &     \\
\min & z_2 = & x_1  & +  & 3x_2 &           &     \\
\text{s.t.} && 2x_1 & +  & 3x_2 & \leq 30   & \qquad (1) \\
            && 3x_1 & +  & 2x_2 & \leq 30   & \qquad(2) \\
            &&  x_1 & -  &  x_2 & \leq 5.5  & \qquad(3) \\
            &&  x_1 & ,\,&  x_2 & \in \mathbb{N} & \qquad (4)
\end{array}
$$

### Données

### Codage du problème MOO avec JuMP et MultiObjectiveAlgorithms

### Sélection du solveur et de l'algorithme

### Extraction de $X_E$ et $Y_N$

### Affichage graphique de $Y_N$

In [ ]:
using Plots
using LaTeXStrings

Z1_opt = []; Z2_opt = []
for i in 1:result_count(model)
    push!(Z1_opt, objective_value(model; result = i)[1])
    push!(Z2_opt, -1 * objective_value(model; result = i)[2])
end

scatter(Z1_opt, Z2_opt,
    title        = L"$Y_N$ (algorithme $\epsilon$-constraint)",
    xlabel       = L"$z_1$ à maximiser",
    ylabel       = L"$z_2$ à minimiser",
    xlims        = (0, 30),
    ylims        = (0, 30),
    aspect_ratio = :equal,
    marker       = :circle,
    markersize   = 5,
    color        = :green,
    legend       = false
)

---
## Exercice : Analyse discriminante et programmation linéaire

**Référence :**   
Ned Freed and Fred Glover. A linear programming approach to the discriminant problem. *Decision Sciences*, volume 12, issue 1, pages 68–74. 1981.

### Énoncé

**But :** Étant donné des observations appartenant à deux groupes **connus**, trouver une règle de classification permettant d'affecter une **nouvelle observation** à l'un des deux groupes.

**Exemple (Freed & Glover, 1981) :** 10 employés, 2 caractéristiques ($x_1$ = expérience, $x_2$ = formation).

| j | $x_1$ | $x_2$ | Groupe  |
|---|-------|-------|----------|
| 1 | 4 | 1 | Succès |
| 2 | 5 | 2 | Succès |
| 3 | 7 | 2 | Succès |
| 4 | 9 | 1 | Succès |
| 5 | 8 | 4 | Succès |
| 6 | 1 | 1 | Échec |
| 7 | 3 | 1 | Échec |
| 8 | 2 | 2 | Échec |
| 9 | 6 | 3 | Échec |
|10 | 3 | 3 | Échec |

### Approche : hyperplan séparateur

On cherche des poids $w_1, w_2$ et un seuil $c$ tels que :

- $\text{score}(j) = x_{1,j} \times w_1 + x_{2,j} \times w_2$
- $\text{score}(j) \geq c$ pour tout individu du groupe *Succès*
- $\text{score}(j) \leq c$ pour tout individu du groupe *Échec*

On introduit une **marge** $d$ et on maximise cette marge :

- Succès $\Rightarrow$ $\text{score}(j) \geq c + d$
- Échec $\Rightarrow$ $\text{score}(j) \leq c - d$

### Modèle LP

$$
\begin{array}{clrcrl}
\max & d \\\\
\text{s.t.} & x^s_{1,j} w_1 + x^s_{2,j} w_2 & - & d & \geq c & \quad \forall j \in \{1,\ldots,n_S\} \\
            & x^e_{1,j} w_1 + x^e_{2,j} w_2 & + & d & \leq c & \quad \forall j \in \{1,\ldots,n_E\} \\
            & w_1,\, w_2,\, d & \in & \mathbb{R} && \quad \text{(non contraint en signe)}
\end{array}
$$

### (a) Données de l'instance

### (b) Modèle JuMP

### (c) Résolution

### (d) Résolution et extraction des résultats

### (e) Visualisation avec Plots.jl

In [ ]:
using Plots

# Calcul droite séparatrice et marges
x1r      = range(0, 12, length=200)
sep      = (c           .- w1_opt .* x1r) ./ w2_opt
marg_sup = (c + d_opt   .- w1_opt .* x1r) ./ w2_opt
marg_inf = (c - d_opt   .- w1_opt .* x1r) ./ w2_opt

# Affichage points
p = scatter(X_s[:,1], X_s[:,2],
        label="Succès", marker=:circle, ms=7, color=:darkgreen,
        title="Analyse discriminante par programmation linéaire",
        xlabel="x1", ylabel="x2")

scatter!(p, X_e[:,1], X_e[:,2],
        label="Échec", marker=:diamond, ms=7, color=:darkred)

# Affichage droites        
plot!(p, x1r, sep,      lw=2, color=:black, label="Seuil c")
plot!(p, x1r, marg_sup, lw=1, ls=:dash, color=:gray, label="Marge +d")
plot!(p, x1r, marg_inf, lw=1, ls=:dash, color=:gray, label="Marge -d")

display(p)